# Problem

In [ ]:
# Import required libraries for optimal control and numerical analysis
using Pkg
Pkg.activate("..")  # Activate the project environment

using Plots                    # For plotting and visualization
using OptimalControl           # Main package for optimal control problems
using NLPModelsIpopt          # Interface for Ipopt nonlinear solver
using OrdinaryDiffEq          # For solving differential equations
using LinearAlgebra: norm     # For vector norm computations
using MINPACK                 # For nonlinear equation solving (shooting method)
using DifferentiationInterface  # For automatic differentiation
using ForwardDiff             # Forward mode automatic differentiation
using Ipopt, Optimization, OptimizationMOI  # Additional optimization tools

In [ ]:
# Define problem parameters and model functions

# Time and boundary conditions
t0 = 0.                                 # Initial time
vf = 10.                                # Final volume to be achieved
x0 = [0., 0., 0.1, 0.1]                 # Initial state
p = [1., 1., 1., 1., 2., 1.5]           # Parameters

# Model component functions
e(x) = 1 + x[3] + x[4]                  # Energy flow
g(x) = 1                                # Volume flow
f11(x) = 1                              # Resistance 1 dynamics (filtration mode)
f12(x) = - x[3]                         # Resistance 1 dynamics (backwash mode) - negative for cleaning
f21(x) = 2                              # Resistance 2 dynamics (filtration mode)  
f22(x) = - 1.5*x[4]                     # Resistance 2 dynamics (backwash mode)

# Vector fields for the control-affine system
Ff(x) = [e(x);  g(x); f11(x); f21(x)]   # Filtration vector field (u = +1)
Fb(x) = [e(x); -g(x); f12(x); f22(x)]   # Backwash vector field (u = -1)
F0(x) = Ff(x) + Fb(x)                   # Drift vector field (symmetric part)
F1(x) = Ff(x) - Fb(x)                   # Control vector field (antisymmetric part)

# Define the optimal control problem using OptimalControl.jl DSL
ocp = @def begin
    tf ∈ R, variable                    # Free final time
    t ∈ [t0, tf], time                  # Time
    x = (e, v, R1, R2) ∈ R⁴, state      # State
    u ∈ R, control                      # Control
    -1 ≤ u(t) ≤ 1                       # Control's constraint  
    x(t0) == x0                         # Initial state
    v(tf) == vf                         # Final state
    ẋ(t) == F0(x(t)) + u(t)*F1(x(t))    # Dynamics   
    e(tf) → min                         # Cost
end

# Direct

In [ ]:
# Solve the optimal control problem using direct method (collocation)
direct_sol = solve(ocp)
# Plot the solution trajectory showing states, controls, and costates over time
plt_sol = plot(direct_sol, label = "direct")

# Indirect

In [ ]:
# Lift into (x,λ) space
H0 = Lift(F0)
H1 = Lift(F1)

# Lie bracket
H01  = @Lie {H0, H1}
H001 = @Lie {H0, H01}
H101 = @Lie {H1, H01}

# Singular control
us(x, p) = -H001(x, p) / H101(x, p)

# Pseudo-Hamiltonian
H(x,p,u) = H0(x,p) + u*H1(x,p)

# Flows
ϕ0 = Flow(ocp, (x,p,tf) -> -1)
ϕ1 = Flow(ocp, (x,p,tf) -> +1)
ϕs = Flow(ocp, (x,p,tf) -> us(x,p))

# Get direct trajectory
time = time_grid(direct_sol)
x = state(direct_sol)
u = control(direct_sol)
p = costate(direct_sol)
tf = time[end]

# Structure of the solution 
plt = plot(t -> H0(x(t), p(t)), t0, tf, label = "H₀(x(t), p(t))")
plot!(plt, t -> H1(x(t), p(t)), t0, tf, label = "H₁(x(t), p(t))")
plot!(plt, t -> H01(x(t), p(t)), t0, tf, label = "H₀₁(x(t), p(t))")
plot!(plt, [t0, tf], [0, 0], c = :black, ls = :dash, label = nothing)

In [ ]:
# Shooting function
function shoot!(s, ξ)
    pv0, pr10, pr20, t1, t2, tf = ξ
    x1, p1 = ϕ1(t0, x0, [-1, pv0, pr10, pr20], t1)
    x2, p2 = ϕs(t1, x1, p1, t2)
    xf, pf = ϕ1(t2, x2, p2, tf)

    s[1] = xf[2] - vf
    s[2:3] = pf[3:4]
    s[4] = H0(x1, p1)
    s[5] = H1(x1, p1)
    s[6] = H01(x1, p1)
end

# Jacobian of the shooting function
jshoot! = (js, ξ) -> jacobian!(shoot!, similar(ξ), js, AutoForwardDiff(), ξ)

# Initial guess
p0 = p(t0)
η = 1e-3
time_ = time[ u.(time) .≤ 1-η ]
t1 = time_[1]; t2 = time_[end]
ξ = [p0[2:4]..., t1, t2, tf]

# Resolution of S(ξ) = 0
indirect_sol = fsolve(shoot!, jshoot!, ξ)

# Plot
pv0, pr10, pr20, t1, t2, tf = indirect_sol.x
p0 = [-1, pv0, pr10, pr20]
ϕ = ϕ1 * (t1, ϕs) * (t2, ϕ1)
flow_sol = ϕ((t0, tf), x0, p0; saveat=range(t0, tf, 1000))
plot!(plt_sol, flow_sol, label="indirect")

# Turnpike property

In [ ]:
# Constraints
function cons!(dξ, ξ,_)
    R1, R2, u = ξ
    x = [0., 0., R1, R2]
    dx = F0(x) + u*F1(x)
    dξ .= dx[3:4]
end

# Objective 
function obj(ξ,_)
    R1, R2, u = ξ
    x = [0., 0., R1, R2]
    dx = F0(x) + u*F1(x)
    return dx[1] - pv0*dx[2]
end

# Initial guess
x, p = ϕ(t0, x0, p0, (t1 + t2)/2)
u = us(x, p)
ξ = [x[3:4]; u]

# Definition of the optimization problem
optprob = OptimizationFunction(obj, AutoForwardDiff(), cons = cons!)
prob = OptimizationProblem(optprob, ξ, nothing, lcons = [0., 0.], ucons = [0., 0.])
opt_sol = Optimization.solve(prob, Ipopt.Optimizer())

# Plot
st = opt_sol.u
plt_sol = plot(flow_sol, label = "")
plot!(plt_sol, [t0, tf], [st[1], st[1]], subplot = 3, label = "", c = :red)
plot!(plt_sol, [t0, tf], [st[2], st[2]], subplot = 4, label = "", c = :red)
plot!(plt_sol, [t0, tf], [st[3], st[3]], subplot = 9, label = "", c = :red)